In [1]:
# 운동 추천 알고리즘에 사용할 라이브러리 불러오기
import pandas as pd
import numpy as np

In [2]:
# 전처리된 추천 후보 운동 영상 데이터 불러오기
workout_videos = pd.read_csv(

    "data/processed/workout_videos.csv"

)

In [3]:
# 추천 알고리즘에 사용할 운동 영상 데이터 확인
print("추천 후보 영상 수:", len(workout_videos))
print("고유 영상 수:", workout_videos["file_nm"].nunique())

display(workout_videos.head())

추천 후보 영상 수: 731
고유 영상 수: 731


,file_nm,title,file_url,video_length,age_group,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,equipment
0,0AUDLJ08S_00041.mp4,요통 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,2299,공통,0.265957,0.265957,0.0,0.468085,0.00,0.00,"['매트', '짐볼']"
1,0AUDLJ08S_00042.mp4,낙상 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,981,공통,0.250000,0.250000,0.0,0.000000,0.25,0.25,"['의자', '줄사다리']"
2,0AUDLJ08S_00043.mp4,우울증 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,2928,공통,0.279070,0.279070,0.0,0.441860,0.00,0.00,"['공', '스텝박스', '짐볼', '풍선']"
3,0AUDLJ08S_00044.mp4,우울증 예방 운동프로그램(댄스운동 편),http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,371,공통,0.195652,0.195652,0.0,0.608696,0.00,0.00,[]
4,0AUDLJ08S_00045.mp4,고혈압 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,2056,공통,0.176471,0.176471,0.0,0.647059,0.00,0.00,"['매트', '밴드', '의자', '짐볼']"


In [6]:
# 추천 알고리즘에서 사용할 6개 체력요인 정의
fitness_columns = [
    "strength",
    "muscularEndurance",
    "cardiovascularEndurance",
    "flexibility",
    "agility",
    "power"
]

In [8]:
# 실제 운동 영상 데이터로 α 후보별 추천 피드백 루프 시뮬레이션

# --------------------------------------------------
# 1. 시뮬레이션 설정
# --------------------------------------------------

alpha_candidates = [0.20, 0.25, 0.30, 0.40]

weekly_frequencies = [1, 2, 3, 7]

simulation_days = 56
lookback_days = 14
half_life = 7


# --------------------------------------------------
# 2. 사용자 유형 정의
#    1 = 양호 / 2 = 보통 / 3 = 개선 필요
# --------------------------------------------------

user_types = {
    "근력약점형": {
        "strength": 3,
        "muscularEndurance": 2,
        "cardiovascularEndurance": 2,
        "flexibility": 2,
        "agility": 1,
        "power": 1
    },

    "심폐약점형": {
        "strength": 2,
        "muscularEndurance": 2,
        "cardiovascularEndurance": 3,
        "flexibility": 2,
        "agility": 1,
        "power": 1
    },

    "유연성약점형": {
        "strength": 2,
        "muscularEndurance": 2,
        "cardiovascularEndurance": 2,
        "flexibility": 3,
        "agility": 1,
        "power": 1
    },

    "복합약점형": {
        "strength": 3,
        "muscularEndurance": 3,
        "cardiovascularEndurance": 3,
        "flexibility": 2,
        "agility": 2,
        "power": 2
    },

    "균형형": {
        "strength": 2,
        "muscularEndurance": 2,
        "cardiovascularEndurance": 2,
        "flexibility": 2,
        "agility": 2,
        "power": 2
    },
    "미측정포함형": {
    "strength": 2,
    "muscularEndurance": None,
    "cardiovascularEndurance": 3,
    "flexibility": 2,
    "agility": None,
    "power": None
}
}


# --------------------------------------------------
# 3. 국민체력100 수준 → 운동 필요도
# --------------------------------------------------

need_mapping = {
    1: 0.25,
    2: 0.50,
    3: 1.00
}


# --------------------------------------------------
# 4. 주간 운동 빈도에 따라 운동일 생성
# --------------------------------------------------

def make_workout_days(weekly_frequency, total_days):

    weekly_patterns = {
        1: [0],
        2: [0, 3],
        3: [0, 2, 4],
        7: list(range(7))
    }

    pattern = weekly_patterns[weekly_frequency]

    workout_days = []

    for week_start in range(0, total_days, 7):

        for day in pattern:

            workout_day = week_start + day

            if workout_day < total_days:
                workout_days.append(workout_day)

    return workout_days


# --------------------------------------------------
# 5. 최근 14일 운동 로그 → 최근 운동 가중치 계산
# --------------------------------------------------

def calculate_recent_exposure(
    logs,
    current_day,
    alpha
):

    exposure = {
        factor: 0.0
        for factor in fitness_columns
    }

    for log in logs:

        days_ago = current_day - log["day"]

        # 현재 추천 시점 이전의 최근 14일 운동만 반영
        if 1 <= days_ago <= lookback_days:

            # 반감기 7일의 시간 감쇠
            time_weight = (
                0.5 ** (days_ago / half_life)
            )

            # 완료 = 1.0 / 미완료 = 0.5
            completion_weight = (
                1.0
                if log["completed"]
                else 0.5
            )

            # 수행한 운동 영상 조회
            video = workout_videos.loc[
                workout_videos["file_nm"]
                == log["videoId"]
            ].iloc[0]

            # 영상의 체력요인 비중을 최근 운동량에 반영
            for factor in fitness_columns:

                video_weight = video[factor]

                exposure[factor] += (
                    video_weight
                    * time_weight
                    * completion_weight
                )

    # α 적용 후 최대 1.0으로 제한
    for factor in fitness_columns:

        exposure[factor] = min(
            alpha * exposure[factor],
            1.0
        )

    return exposure


# --------------------------------------------------
# 6. 체력 상태 + 최근 운동량 → 추천 우선순위 계산
# --------------------------------------------------

def calculate_priority(
    fitness_data,
    recent_exposure
):

    priority = {}

    for factor in fitness_columns:

        level = fitness_data.get(factor)

        # 측정값이 없는 체력요인
        if level is None:

            priority[factor] = (
                1.5
                * (1 - recent_exposure[factor])
            )

        # 측정값이 있는 체력요인
        else:

            need = need_mapping[level]

            priority[factor] = (
                need
                + (1 - recent_exposure[factor])
            )

    return priority


# --------------------------------------------------
# 7. 체력요인 우선순위로 영상별 추천 점수 계산
# --------------------------------------------------

def recommend_video(priority):

    scores = np.zeros(
        len(workout_videos)
    )

    for factor in fitness_columns:

        scores += (
            workout_videos[factor].to_numpy()
            * priority[factor]
        )

    best_index = np.argmax(scores)

    return workout_videos.iloc[best_index]


# --------------------------------------------------
# 8. 사용자 1명의 추천 과정을 반복 시뮬레이션
# --------------------------------------------------

def simulate_user(
    fitness_data,
    weekly_frequency,
    alpha
):

    workout_days = make_workout_days(
        weekly_frequency,
        simulation_days
    )

    logs = []

    saturation_count = 0
    exposure_check_count = 0

    max_exposure_values = []

    recommendation_counts = {
        factor: 0
        for factor in fitness_columns
    }

    # 3등급 체력요인을 약점으로 정의
    weak_factors = [
        factor
        for factor, level
        in fitness_data.items()
        if level == 3
    ]

    weak_recommendation_count = 0

    for current_day in range(
        simulation_days
    ):

        # 운동하지 않는 날
        if current_day not in workout_days:
            continue

        # 최근 운동량 계산
        recent_exposure = (
            calculate_recent_exposure(
                logs,
                current_day,
                alpha
            )
        )

        # 체력요인별 포화 여부 기록
        for factor in fitness_columns:

            exposure_check_count += 1

            if recent_exposure[factor] >= 1.0:
                saturation_count += 1

        # 해당 운동일의 가장 높은 가중치 기록
        max_exposure_values.append(
            max(recent_exposure.values())
        )

        # 현재 추천 우선순위 계산
        priority = calculate_priority(
            fitness_data,
            recent_exposure
        )

        # 가장 점수가 높은 영상 추천
        video = recommend_video(
            priority
        )

        # 추천 영상에서 비중이 가장 큰 체력요인
        video_weights = {
            factor: video[factor]
            for factor in fitness_columns
        }

        main_factor = max(
            video_weights,
            key=video_weights.get
        )

        recommendation_counts[
            main_factor
        ] += 1

        # 추천 영상에 약점 체력요인이 포함되는지 확인
        if weak_factors:

            if any(
                video[factor] > 0
                for factor in weak_factors
            ):
                weak_recommendation_count += 1

        # 추천 영상을 완료했다고 가정하고 로그 저장
        logs.append({
            "day": current_day,
            "videoId": video["file_nm"],
            "completed": True
        })

    total_workouts = len(logs)

    # 포화율
    if exposure_check_count > 0:

        saturation_rate = (
            saturation_count
            / exposure_check_count
        )

    else:
        saturation_rate = 0

    # 약점 종목 추천 비율
    if (
        total_workouts > 0
        and len(weak_factors) > 0
    ):

        weak_ratio = (
            weak_recommendation_count
            / total_workouts
        )

    else:
        weak_ratio = np.nan

    # 시뮬레이션 중 관측된 최대 가중치
    if max_exposure_values:

        max_weight = max(
            max_exposure_values
        )

    else:
        max_weight = 0

    return {
        "total_workouts": total_workouts,
        "saturation_rate": saturation_rate,
        "max_weight": max_weight,
        "weak_ratio": weak_ratio,

        **{
            f"{factor}_count": count
            for factor, count
            in recommendation_counts.items()
        }
    }


# --------------------------------------------------
# 9. α × 사용자 유형 × 운동 빈도 전체 실행
# --------------------------------------------------

simulation_results = []

for alpha in alpha_candidates:

    for user_type, fitness_data in (
        user_types.items()
    ):

        for weekly_frequency in (
            weekly_frequencies
        ):

            result = simulate_user(
                fitness_data,
                weekly_frequency,
                alpha
            )

            simulation_results.append({
                "alpha": alpha,
                "user_type": user_type,
                "weekly_frequency":
                    weekly_frequency,
                **result
            })


simulation_results_df = pd.DataFrame(
    simulation_results
)


# --------------------------------------------------
# 10. α별 핵심 결과 요약
# --------------------------------------------------

alpha_summary = (
    simulation_results_df
    .groupby("alpha")
    .agg(
        saturation_rate=(
            "saturation_rate",
            "mean"
        ),
        average_max_weight=(
            "max_weight",
            "mean"
        ),
        average_weak_ratio=(
            "weak_ratio",
            "mean"
        )
    )
    .reset_index()
)




# 비율을 % 단위로 변환
alpha_summary["saturation_rate"] = (
    alpha_summary["saturation_rate"]
    * 100
).round(2)

alpha_summary["average_weak_ratio"] = (
    alpha_summary["average_weak_ratio"]
    * 100
).round(2)

alpha_summary["average_max_weight"] = (
    alpha_summary["average_max_weight"]
    .round(3)
)

alpha_summary

,alpha,saturation_rate,average_max_weight,average_weak_ratio
0,0.20,0.00,0.394,86.55
1,0.25,0.00,0.464,83.42
2,0.30,0.01,0.529,77.02
3,0.40,1.18,0.616,70.70


In [9]:
simulation_results_df[
    (simulation_results_df["user_type"] == "미측정포함형")
    & (simulation_results_df["alpha"] == 0.30)
]

,alpha,user_type,weekly_frequency,total_workouts,saturation_rate,max_weight,weak_ratio,strength_count,muscularEndurance_count,cardiovascularEndurance_count,flexibility_count,agility_count,power_count
68,0.3,미측정포함형,1,8,0.0,0.225000,1.000000,0,0,8,0,0,0
69,0.3,미측정포함형,2,16,0.0,0.559349,0.812500,3,0,13,0,0,0
70,0.3,미측정포함형,3,24,0.0,0.646036,0.625000,4,0,15,3,2,0
71,0.3,미측정포함형,7,56,0.0,0.989269,0.392857,14,0,22,10,10,0


In [10]:
# 추천 알고리즘에서 사용할 기본 설정값 정의
fitness_columns = [
    "strength",
    "muscularEndurance",
    "cardiovascularEndurance",
    "flexibility",
    "agility",
    "power"
]

EXPOSURE_ALPHA = 0.30
LOOKBACK_DAYS = 14
HALF_LIFE = 7

MISSING_PRIORITY = 1.50
RECENT_VIDEO_DAYS = 7

In [11]:
# 국민체력100 중첩 데이터에서 체력요인 측정값 추출
def extract_fitness_data(fitness100):
    return (fitness100 or {}).get("fitness") or {}


test_fitness100 = {
    "bodyComposition": {
        "height": 170,
        "weight": 65
    },
    "fitness": {
        "strength": 2,
        "muscularEndurance": None,
        "cardiovascularEndurance": 3,
        "flexibility": 2,
        "agility": None,
        "power": None
    }
}

test_extracted_fitness = extract_fitness_data(
    test_fitness100
)

assert test_extracted_fitness["strength"] == 2
assert (
    test_extracted_fitness["cardiovascularEndurance"]
    == 3
)

test_extracted_fitness

{'strength': 2,
 'muscularEndurance': None,
 'cardiovascularEndurance': 3,
 'flexibility': 2,
 'agility': None,
 'power': None}

In [12]:
# 오늘 운동 수행 후 다음 운동 추천에 사용할 기준 날짜 계산
def get_next_recommendation_date(current_date):
    current_date = pd.to_datetime(current_date)
    return current_date + pd.Timedelta(days=1)


current_date = "2026-09-23"
next_recommendation_date = get_next_recommendation_date(
    current_date
)

workout_date = pd.to_datetime("2026-09-23")
days_ago = (
    next_recommendation_date - workout_date
).days

assert next_recommendation_date == pd.Timestamp(
    "2026-09-24 00:00:00"
)
assert days_ago == 1

next_recommendation_date, days_ago

(Timestamp('2026-09-24 00:00:00'), 1)

In [13]:
# 최근 14일 운동 로그를 기반으로 체력요인별 최근 운동 노출도 계산
def calculate_recent_exposure(logs, current_date):

    exposure = {
        factor: 0.0
        for factor in fitness_columns
    }

    current_date = pd.to_datetime(current_date)

    for log in logs:

        workout_date = pd.to_datetime(log["date"])
        days_ago = (current_date - workout_date).days

        # 추천일 이전 최근 14일 운동만 반영
        if not (1 <= days_ago <= LOOKBACK_DAYS):
            continue

        # 오래된 운동일수록 영향 감소
        time_weight = 0.5 ** (
            days_ago / HALF_LIFE
        )

        # 완료 여부에 따른 가중치
        completion_weight = (
            1.0 if log["completed"] else 0.5
        )

        # 운동 로그에 해당하는 영상 조회
        matched_video = workout_videos[
            workout_videos["file_nm"] == log["videoId"]
        ]

        # 현재 영상 데이터에 존재하지 않는 로그는 제외
        if matched_video.empty:
            continue

        video = matched_video.iloc[0]

        # 영상이 가진 체력요인 비율을 누적
        for factor in fitness_columns:

            exposure[factor] += (
                video[factor]
                * time_weight
                * completion_weight
            )

    # α를 적용하고 각 체력요인의 최대값을 1로 제한
    for factor in fitness_columns:

        exposure[factor] = min(
            EXPOSURE_ALPHA * exposure[factor],
            1.0
        )

    return exposure

In [14]:
# 체력등급을 추천 우선순위 계산에 사용할 운동 필요도로 변환
def get_fitness_need(level):

    if level is None or isinstance(level, bool):
        return None

    if level == 1:
        return 0.25

    if level == 2:
        return 0.50

    if (
        isinstance(level, (int, float))
        and level >= 3
    ):
        return 1.00

    return None

In [15]:
# 체력 수준과 최근 운동 노출도를 결합해 체력요인별 최종 추천 우선순위 계산
def calculate_priority(fitness_data, recent_exposure):

    priority = {}

    for factor in fitness_columns:

        level = fitness_data.get(factor)

        need = get_fitness_need(level)

        # 미측정 또는 유효하지 않은 체력요인
        if need is None:
            priority[factor] = (
                MISSING_PRIORITY
                * (1 - recent_exposure[factor])
            )

        # 측정된 체력요인
        else:
            priority[factor] = (
                need
                + (1 - recent_exposure[factor])
            )

    return priority

In [16]:
# 체력등급별 운동 필요도와 4등급 우선순위 계산 테스트
test_need_results = {
    "1": get_fitness_need(1),
    "2": get_fitness_need(2),
    "3": get_fitness_need(3),
    "4": get_fitness_need(4),
    "5": get_fitness_need(5),
    "6": get_fitness_need(6),
    "None": get_fitness_need(None),
    "string_3": get_fitness_need("3"),
    "True": get_fitness_need(True)
}

assert test_need_results == {
    "1": 0.25,
    "2": 0.50,
    "3": 1.00,
    "4": 1.00,
    "5": 1.00,
    "6": 1.00,
    "None": None,
    "string_3": None,
    "True": None
}

test_fitness_data_grade_4 = {
    factor: None
    for factor in fitness_columns
}
test_fitness_data_grade_4["strength"] = 4

test_zero_exposure = {
    factor: 0.0
    for factor in fitness_columns
}

test_priority_grade_4 = calculate_priority(
    test_fitness_data_grade_4,
    test_zero_exposure
)

assert test_priority_grade_4["strength"] == 2.0

{
    "need_results": test_need_results,
    "priority_with_grade_4": test_priority_grade_4
}

{'need_results': {'1': 0.25,
  '2': 0.5,
  '3': 1.0,
  '4': 1.0,
  '5': 1.0,
  '6': 1.0,
  'None': None,
  'string_3': None,
  'True': None},
 'priority_with_grade_4': {'strength': 2.0,
  'muscularEndurance': 1.5,
  'cardiovascularEndurance': 1.5,
  'flexibility': 1.5,
  'agility': 1.5,
  'power': 1.5}}

In [17]:
# 운동 영상 데이터에 존재하는 연령 그룹 확인
workout_videos["age_group"].value_counts(dropna=False)

age_group
공통     322
청소년    182
어르신    125
유소년     97
성인       3
유아기      2
Name: count, dtype: int64

In [18]:
# 청소년·성인 추천 후보의 체력요인별 영상 수 확인
for target_group in ["청소년", "성인"]:

    candidates = workout_videos[
        workout_videos["age_group"].isin(
            ["공통", target_group]
        )
    ]

    print(f"\n[{target_group}] 총 후보: {len(candidates)}개")

    for factor in fitness_columns:
        count = (candidates[factor] > 0).sum()
        print(f"{factor}: {count}개")


[청소년] 총 후보: 504개
strength: 237개
muscularEndurance: 64개
cardiovascularEndurance: 32개
flexibility: 164개
agility: 58개
power: 60개

[성인] 총 후보: 325개
strength: 190개
muscularEndurance: 16개
cardiovascularEndurance: 32개
flexibility: 81개
agility: 34개
power: 34개


In [19]:
# 사용자 나이에 맞는 연령 그룹의 운동 영상만 추천 후보로 필터링
def filter_by_age(workout_videos, age):

    if age is None:
        raise ValueError(
            "사용자 나이 정보가 필요합니다."
        )

    # 청소년기
    if 13 <= age <= 18:
        target_groups = ["공통", "청소년"]

    # 성인기
    elif 19 <= age <= 64:
        target_groups = ["공통", "성인"]

    # 어르신
    elif age >= 65:
        target_groups = ["어르신"]

    # 유소년
    else:
        target_groups = ["유소년"]

    candidates = workout_videos[
        workout_videos["age_group"].isin(
            target_groups
        )
    ].copy()

    return candidates

In [20]:
# 최근 7일 동안 수행한 운동 영상을 추천 후보에서 제외
def exclude_recent_videos(candidates, logs, current_date):

    current_date = pd.to_datetime(current_date)

    recent_file_names = []

    for log in logs:

        workout_date = pd.to_datetime(log["date"])
        days_ago = (current_date - workout_date).days

        # 추천일 이전 최근 7일의 영상 식별자 저장
        if 1 <= days_ago <= RECENT_VIDEO_DAYS:
            recent_file_names.append(
                log["videoId"]
            )

    filtered_candidates = candidates[
        ~candidates["file_nm"].isin(
            recent_file_names
        )
    ].copy()

    return filtered_candidates

In [21]:
# 후보 영상마다 체력요인 우선순위와 영상 비율을 결합해 추천 점수 계산
def calculate_video_scores(candidates, priority):

    scored_candidates = candidates.copy()

    scored_candidates["recommendation_score"] = 0.0

    for factor in fitness_columns:

        scored_candidates["recommendation_score"] += (
            scored_candidates[factor]
            * priority[factor]
        )

    return scored_candidates

In [22]:
# 최고점 영상 중 최근 수행 이력이 가장 오래된 영상을 우선 선택
def select_best_video(scored_candidates, logs):

    if scored_candidates.empty:
        raise ValueError(
            "추천 가능한 운동 영상이 없습니다."
        )

    max_score = scored_candidates[
        "recommendation_score"
    ].max()

    # 부동소수점 오차를 고려해 최고점 후보 선택
    best_candidates = scored_candidates[
        np.isclose(
            scored_candidates["recommendation_score"],
            max_score
        )
    ].copy()

    # 영상별 가장 최근 수행일 계산
    last_workout_dates = {}

    for log in logs:
        file_nm = log["videoId"]
        workout_date = pd.to_datetime(log["date"])

        if (
            file_nm not in last_workout_dates
            or workout_date > last_workout_dates[file_nm]
        ):
            last_workout_dates[file_nm] = workout_date

    best_candidates["last_workout_date"] = (
        best_candidates["file_nm"]
        .map(last_workout_dates)
    )

    # 한 번도 수행하지 않은 최고점 영상 우선
    never_used = best_candidates[
        best_candidates["last_workout_date"].isna()
    ]

    if not never_used.empty:
        selected_video = never_used.sample(n=1).iloc[0]

    else:
        # 모두 수행했다면 가장 오래전에 수행한 영상 우선
        oldest_date = best_candidates[
            "last_workout_date"
        ].min()

        oldest_candidates = best_candidates[
            best_candidates["last_workout_date"]
            == oldest_date
        ]

        selected_video = oldest_candidates.sample(n=1).iloc[0]

    return selected_video

In [23]:
# 사용자 체력정보와 운동기록을 기반으로 다음 운동 영상 1개 추천
def recommend_next_workout(
    age,
    fitness_data,
    logs,
    current_date
):

    # 1. 최근 운동 노출도 계산
    recent_exposure = calculate_recent_exposure(
        logs,
        current_date
    )

    # 2. 체력요인별 추천 우선순위 계산
    priority = calculate_priority(
        fitness_data,
        recent_exposure
    )

    # 3. 연령 조건에 맞는 후보 생성
    age_candidates = filter_by_age(
        workout_videos,
        age
    )

    # 4. 최근 7일 수행 영상 제외
    candidates = exclude_recent_videos(
        age_candidates,
        logs,
        current_date
    )

    # 5. 후보가 없으면 최근 7일 중복 제외 규칙만 해제
    if candidates.empty:
        candidates = age_candidates.copy()

    # 6. 영상별 추천 점수 계산
    scored_candidates = calculate_video_scores(
        candidates,
        priority
    )

    # 7. 최고점 영상 중 수행 이력을 고려해 최종 선택
    selected_video = select_best_video(
        scored_candidates,
        logs
    )

    return {
        "videoId": selected_video["file_nm"]
    }

In [24]:
# 운동 이력이 없는 신규 사용자의 첫 운동 추천 테스트
test_fitness = {
    "strength": 2,
    "muscularEndurance": 3,
    "cardiovascularEndurance": 2,
    "flexibility": 1,
    "agility": None,
    "power": None
}

test_logs = []

test_result = recommend_next_workout(
    age=25,
    fitness_data=test_fitness,
    logs=test_logs,
    current_date="2026-09-23"
)

test_result

{'videoId': '0AUDLJ08S_00047.mp4'}

In [25]:
# 첫 추천 영상의 체력요인 구성과 추천 점수 및 동점 후보 확인
test_recent_exposure = calculate_recent_exposure(
    test_logs,
    "2026-09-23"
)

test_priority = calculate_priority(
    test_fitness,
    test_recent_exposure
)

test_candidates = filter_by_age(
    workout_videos,
    25
)

test_candidates = exclude_recent_videos(
    test_candidates,
    test_logs,
    "2026-09-23"
)

test_scored = calculate_video_scores(
    test_candidates,
    test_priority
)

selected_file_nm = test_result["videoId"]

selected_video = test_scored[
    test_scored["file_nm"] == selected_file_nm
]

max_score = test_scored["recommendation_score"].max()

top_candidates = test_scored[
    np.isclose(
        test_scored["recommendation_score"],
        max_score
    )
]

print("체력요인별 priority")
print(test_priority)

print("\n추천된 영상")
display(
    selected_video[
        [
            "file_nm",
            "title",
            "age_group",
            *fitness_columns,
            "recommendation_score"
        ]
    ]
)

print(f"\n최고 추천 점수: {max_score:.4f}")
print(f"최고점 동점 후보 수: {len(top_candidates)}개")

display(
    top_candidates[
        [
            "file_nm",
            "title",
            "age_group",
            *fitness_columns,
            "recommendation_score"
        ]
    ]
)

체력요인별 priority
{'strength': 1.5, 'muscularEndurance': 2.0, 'cardiovascularEndurance': 1.5, 'flexibility': 1.25, 'agility': 1.5, 'power': 1.5}

추천된 영상


,file_nm,title,age_group,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,recommendation_score
6,0AUDLJ08S_00047.mp4,골다공증 예방 운동프로그램,공통,0.5,0.5,0.0,0.0,0.0,0.0,1.75



최고 추천 점수: 1.7500
최고점 동점 후보 수: 4개


,file_nm,title,age_group,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,recommendation_score
6,0AUDLJ08S_00047.mp4,골다공증 예방 운동프로그램,공통,0.5,0.5,0.0,0.0,0.0,0.0,1.75
9,0AUDLJ08S_00050.mp4,짐볼을 활용한 근력운동 프로그램,공통,0.5,0.5,0.0,0.0,0.0,0.0,1.75
11,0AUDLJ08S_00052.mp4,품롤러를 활용한 근력운동 프로그램,공통,0.5,0.5,0.0,0.0,0.0,0.0,1.75
724,0AUDLJ08S_01014.mp4,냄비를 활용한 유산소 전신 근력 운동,공통,0.5,0.5,0.0,0.0,0.0,0.0,1.75


In [26]:
# 최근 근력·근지구력 운동 이력이 쌓였을 때 추천 변화 확인
test_logs_after_workout = [
    {
        "date": "2026-09-22",
        "videoId": "0AUDLJ08S_00047.mp4",
        "completed": True
    },
    {
        "date": "2026-09-20",
        "videoId": "0AUDLJ08S_00050.mp4",
        "completed": True
    },
    {
        "date": "2026-09-18",
        "videoId": "0AUDLJ08S_00052.mp4",
        "completed": True
    }
]

test_exposure_after = calculate_recent_exposure(
    test_logs_after_workout,
    "2026-09-23"
)

test_priority_after = calculate_priority(
    test_fitness,
    test_exposure_after
)

test_result_after = recommend_next_workout(
    age=25,
    fitness_data=test_fitness,
    logs=test_logs_after_workout,
    current_date="2026-09-23"
)

print("운동 전 priority")
print(test_priority)

print("\n최근 운동 노출도")
print(test_exposure_after)

print("\n운동 후 priority")
print(test_priority_after)

print("\n다음 추천")
print(test_result_after)

운동 전 priority
{'strength': 1.5, 'muscularEndurance': 2.0, 'cardiovascularEndurance': 1.5, 'flexibility': 1.25, 'agility': 1.5, 'power': 1.5}

최근 운동 노출도
{'strength': np.float64(0.33873414539019275), 'muscularEndurance': np.float64(0.33873414539019275), 'cardiovascularEndurance': np.float64(0.0), 'flexibility': np.float64(0.0), 'agility': np.float64(0.0), 'power': np.float64(0.0)}

운동 후 priority
{'strength': np.float64(1.1612658546098071), 'muscularEndurance': np.float64(1.6612658546098071), 'cardiovascularEndurance': np.float64(1.5), 'flexibility': np.float64(1.25), 'agility': np.float64(1.5), 'power': np.float64(1.5)}

다음 추천
{'videoId': '0AUDLJ08S_00318.mp4'}


In [27]:
# 운동 이력 반영 후 추천 근거와 최근 7일 영상 제외 여부 확인
age_candidates = filter_by_age(
    workout_videos,
    25
)

filtered_candidates = exclude_recent_videos(
    age_candidates,
    test_logs_after_workout,
    "2026-09-23"
)

scored_candidates = calculate_video_scores(
    filtered_candidates,
    test_priority_after
)

selected_file_nm = test_result_after["videoId"]

selected_video = scored_candidates[
    scored_candidates["file_nm"] == selected_file_nm
]

recent_files = [
    log["videoId"]
    for log in test_logs_after_workout
]

print("추천된 영상")
display(
    selected_video[
        [
            "file_nm",
            "title",
            "age_group",
            *fitness_columns,
            "recommendation_score"
        ]
    ]
)

print("연령 필터 후 후보 수:", len(age_candidates))
print("최근 7일 제외 후 후보 수:", len(filtered_candidates))

print("\n최근 운동 영상의 후보 포함 여부")

for file_nm in recent_files:
    is_in_candidates = (
        file_nm in filtered_candidates["file_nm"].values
    )

    print(
        file_nm,
        "→",
        "포함" if is_in_candidates else "제외"
    )

추천된 영상


,file_nm,title,age_group,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,recommendation_score
102,0AUDLJ08S_00318.mp4,계단 두 칸씩 두발 뛰기,공통,0.0,0.0,0.0,0.0,0.5,0.5,1.5


연령 필터 후 후보 수: 325
최근 7일 제외 후 후보 수: 322

최근 운동 영상의 후보 포함 여부
0AUDLJ08S_00047.mp4 → 제외
0AUDLJ08S_00050.mp4 → 제외
0AUDLJ08S_00052.mp4 → 제외


In [28]:
# 일부 체력요인이 미측정된 사용자의 추천 우선순위와 추천 결과 검증
partial_fitness = {
    "strength": 2,
    "muscularEndurance": None,
    "cardiovascularEndurance": 3,
    "flexibility": 2,
    "agility": None,
    "power": None
}

partial_logs = []

partial_exposure = calculate_recent_exposure(
    partial_logs,
    "2026-09-23"
)

partial_priority = calculate_priority(
    partial_fitness,
    partial_exposure
)

partial_result = recommend_next_workout(
    age=25,
    fitness_data=partial_fitness,
    logs=partial_logs,
    current_date="2026-09-23"
)

print("최근 운동 노출도")
print(partial_exposure)

print("\n체력요인별 priority")
print(partial_priority)

print("\n추천 결과")
print(partial_result)

최근 운동 노출도
{'strength': 0.0, 'muscularEndurance': 0.0, 'cardiovascularEndurance': 0.0, 'flexibility': 0.0, 'agility': 0.0, 'power': 0.0}

체력요인별 priority
{'strength': 1.5, 'muscularEndurance': 1.5, 'cardiovascularEndurance': 2.0, 'flexibility': 1.5, 'agility': 1.5, 'power': 1.5}

추천 결과
{'videoId': '0AUDLJ08S_00184.mp4'}


In [29]:
# 실제 추천된 영상의 추천 점수와 전체 최고 점수를 비교
selected_file = test_result["videoId"]

test_exposure = calculate_recent_exposure(
    test_logs,
    "2026-09-23"
)

test_priority = calculate_priority(
    test_fitness,
    test_exposure
)

test_candidates = filter_by_age(
    workout_videos,
    25
)

test_candidates = exclude_recent_videos(
    test_candidates,
    test_logs,
    "2026-09-23"
)

test_scored = calculate_video_scores(
    test_candidates,
    test_priority
)

selected_video = test_scored[
    test_scored["file_nm"] == selected_file
]

selected_score = selected_video[
    "recommendation_score"
].iloc[0]

max_score = test_scored[
    "recommendation_score"
].max()

print("추천 영상")
display(
    selected_video[
        [
            "file_nm",
            "title",
            "strength",
            "muscularEndurance",
            "cardiovascularEndurance",
            "flexibility",
            "agility",
            "power",
            "recommendation_score"
        ]
    ]
)

print("추천 영상 점수:", selected_score)
print("전체 최고 점수:", max_score)
print(
    "최고점 후보 여부:",
    np.isclose(selected_score, max_score)
)

추천 영상


,file_nm,title,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,recommendation_score
6,0AUDLJ08S_00047.mp4,골다공증 예방 운동프로그램,0.5,0.5,0.0,0.0,0.0,0.0,1.75


추천 영상 점수: 1.75
전체 최고 점수: 1.75
최고점 후보 여부: True


In [30]:
# 오늘 운동 반영 전후의 체력요인별 내부 가중치 변화를 계산
def calculate_weight_adjustment(
    logs,
    current_date
):

    current_date = pd.to_datetime(current_date)

    # 오늘 운동 반영 전: 기존 1~14일 운동 노출도
    previous = calculate_recent_exposure(
        logs,
        current_date
    )

    # previous를 복사하여 next 계산 시작
    next_weights = previous.copy()

    # 오늘 수행한 운동 로그만 반영
    for log in logs:

        workout_date = pd.to_datetime(log["date"])
        days_ago = (current_date - workout_date).days

        if days_ago != 0:
            continue

        matched_video = workout_videos[
            workout_videos["file_nm"]
            == log["videoId"]
        ]

        if matched_video.empty:
            continue

        video = matched_video.iloc[0]

        completion_weight = (
            1.0 if log["completed"] else 0.5
        )

        for factor in fitness_columns:

            next_weights[factor] += (
                EXPOSURE_ALPHA
                * video[factor]
                * completion_weight
            )

    # 노출도 상한 1.0 적용
    for factor in fitness_columns:
        next_weights[factor] = min(
            next_weights[factor],
            1.0
        )

    # 개발자 전달용 WeightAdjustment 생성
    adjustment = {}

    for factor in fitness_columns:

        adjustment[factor] = {
            "previous": round(
                previous[factor], 3
            ),
            "delta": round(
                next_weights[factor]
                - previous[factor],
                3
            ),
            "next": round(
                next_weights[factor],
                3
            )
        }

    return adjustment

In [31]:
# 개발자 입력을 받아 다음 운동 추천과 내부 가중치 변화를 함께 반환
def run_recommendation(
    profile,
    fitness100,
    logs,
    current_date
):

    age = profile["age"]

    fitness_data = extract_fitness_data(
        fitness100
    )

    weight_adjustment = calculate_weight_adjustment(
        logs,
        current_date
    )

    next_date = get_next_recommendation_date(
        current_date
    )

    next_workout = recommend_next_workout(
        age,
        fitness_data,
        logs,
        next_date
    )

    return {
        "nextWorkout": next_workout,
        "weightAdjustment": weight_adjustment
    }


test_profile = {
    "age": 30,
    "sex": "female"
}

test_fitness100 = {
    "bodyComposition": {
        "height": 165,
        "weight": 55
    },
    "fitness": {
        "strength": 2,
        "muscularEndurance": 3,
        "cardiovascularEndurance": 2,
        "flexibility": 1,
        "agility": None,
        "power": None
    }
}

test_logs = []
current_date = "2026-09-23"

test_recommendation_result = run_recommendation(
    test_profile,
    test_fitness100,
    test_logs,
    current_date
)

assert set(test_recommendation_result) == {
    "nextWorkout",
    "weightAdjustment"
}
assert set(test_recommendation_result["nextWorkout"]) == {
    "videoId"
}
assert set(
    test_recommendation_result["weightAdjustment"]
) == set(fitness_columns)

for adjustment in test_recommendation_result[
    "weightAdjustment"
].values():
    assert set(adjustment) == {
        "previous",
        "delta",
        "next"
    }

test_recommendation_result

{'nextWorkout': {'videoId': '0AUDLJ08S_00052.mp4'},
 'weightAdjustment': {'strength': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
  'muscularEndurance': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
  'cardiovascularEndurance': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
  'flexibility': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
  'agility': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
  'power': {'previous': 0.0, 'delta': 0.0, 'next': 0.0}}}

In [32]:
# 오늘 운동이 없을 때 모든 체력요인의 delta가 0인지 확인
test_logs_no_today = []

test_adjustment = calculate_weight_adjustment(
    logs=test_logs_no_today,
    current_date="2026-09-23"
)

test_adjustment

{'strength': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'muscularEndurance': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'cardiovascularEndurance': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'flexibility': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'agility': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'power': {'previous': 0.0, 'delta': 0.0, 'next': 0.0}}

In [33]:
# 동일한 오늘 운동의 완료·미완료에 따른 delta 차이 확인
test_file_nm = test_result["videoId"]

completed_logs = [
    {
        "date": "2026-09-23",
        "videoId": test_file_nm,
        "completed": True
    }
]

incomplete_logs = [
    {
        "date": "2026-09-23",
        "videoId": test_file_nm,
        "completed": False
    }
]

completed_adjustment = calculate_weight_adjustment(
    logs=completed_logs,
    current_date="2026-09-23"
)

incomplete_adjustment = calculate_weight_adjustment(
    logs=incomplete_logs,
    current_date="2026-09-23"
)

for factor in fitness_columns:
    print(
        factor,
        "| 완료:",
        completed_adjustment[factor]["delta"],
        "| 미완료:",
        incomplete_adjustment[factor]["delta"]
    )

strength | 완료: 0.15 | 미완료: 0.075
muscularEndurance | 완료: 0.15 | 미완료: 0.075
cardiovascularEndurance | 완료: 0.0 | 미완료: 0.0
flexibility | 완료: 0.0 | 미완료: 0.0
agility | 완료: 0.0 | 미완료: 0.0
power | 완료: 0.0 | 미완료: 0.0


In [34]:
# 오늘 서로 다른 운동 2개를 수행했을 때 delta가 합산되는지 확인
strength_video = workout_videos[
    (workout_videos["strength"] > 0)
    & (workout_videos["muscularEndurance"] > 0)
].iloc[0]

cardio_video = workout_videos[
    workout_videos["cardiovascularEndurance"] == 1.0
].iloc[0]

two_workout_logs = [
    {
        "date": "2026-09-23",
        "videoId": strength_video["file_nm"],
        "completed": True
    },
    {
        "date": "2026-09-23",
        "videoId": cardio_video["file_nm"],
        "completed": True
    }
]

two_workout_adjustment = calculate_weight_adjustment(
    logs=two_workout_logs,
    current_date="2026-09-23"
)

print("운동 1:", strength_video["title"])
print("운동 2:", cardio_video["title"])
print()

for factor in fitness_columns:
    print(
        factor,
        "| delta:",
        two_workout_adjustment[factor]["delta"]
    )

운동 1: 요통 예방 운동프로그램
운동 2: 실내 자전거타기

strength | delta: 0.08
muscularEndurance | delta: 0.08
cardiovascularEndurance | delta: 0.3
flexibility | delta: 0.14
agility | delta: 0.0
power | delta: 0.0


In [35]:
# 테스트 ③에 사용한 두 영상의 실제 체력요인 가중치 확인
display(
    workout_videos[
        workout_videos["file_nm"].isin([
            strength_video["file_nm"],
            cardio_video["file_nm"]
        ])
    ][
        [
            "file_nm",
            "title",
            "strength",
            "muscularEndurance",
            "cardiovascularEndurance",
            "flexibility",
            "agility",
            "power"
        ]
    ]
)

,file_nm,title,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power
0,0AUDLJ08S_00041.mp4,요통 예방 운동프로그램,0.265957,0.265957,0.0,0.468085,0.0,0.0
18,0AUDLJ08S_00173.mp4,실내 자전거타기,0.000000,0.000000,1.0,0.000000,0.0,0.0


In [ ]:
# 오늘 수행한 영상이 다음 추천에서 제외되고 내부 가중치에 반영되는지 검증
today_video = test_recommendation_result["nextWorkout"]["videoId"]

today_logs = [
    {
        "date": "2026-09-23",
        "videoId": today_video,
        "completed": True
    }
]

today_result = run_recommendation(
    test_profile,
    test_fitness100,
    today_logs,
    "2026-09-23"
)

assert today_result["nextWorkout"]["videoId"] != today_video

assert any(
    change["delta"] > 0
    for change in today_result["weightAdjustment"].values()
)

today_result